# Module 05 — Cross-Dataset Integration

This notebook summarises the cross-dataset integration produced by
`scripts/05_integration.py`. The 12 IVD scRNA-seq datasets were integrated
using a **tiered strategy**:

- **Tier 1 — Non-resident cells** (immune, endothelial, pericyte): standard
  scVI integration with `batch_key='study'`.
- **Tier 2 — Resident IVD cells**: NP and AF compartments integrated
  separately, benchmarked across four approaches:
  - **A** — scVI (conservative, n_latent=20, compartment covariate)
  - **B** — scANVI (semi-supervised from A, high-confidence labels as seeds)
  - **C** — Harmony (best theta from 0.5 / 1.0 / 2.0)
  - **D** — BBKNN (graph-level batch correction)

EP cells (155) are included with NP; fibroblasts (1,862) with AF; "other" (684)
are excluded.

**Manuscript mapping:** Supplementary Figure S3 — integration benchmarking.
Methods section on integration strategy and rationale for chosen approach.

**Data sources:**
- `data/integrated/tier1_nonresident.h5ad`
- `data/integrated/tier2_resident_NP.h5ad`
- `data/integrated/tier2_resident_AF.h5ad`
- `results/integration/integration_metrics.tsv`

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='scanpy')

from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 200, 'savefig.bbox': 'tight'})

# ── Paths ──────────────────────────────────────────────────────────────────
BASE = Path('..').resolve()
INT_DIR = BASE / 'data' / 'integrated'
RESULTS_DIR = BASE / 'results' / 'integration'
FIG_DIR = RESULTS_DIR
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Load metrics ──────────────────────────────────────────────────────────
metrics_path = RESULTS_DIR / 'integration_metrics.tsv'
if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path, sep='\t')
    print(f'Loaded metrics: {len(metrics_df)} rows')
    display(metrics_df)
else:
    metrics_df = pd.DataFrame()
    print('WARNING: integration_metrics.tsv not found — run the script first')

## Tier 1 — Non-Resident Cells

Non-resident cells (immune, endothelial, pericyte; ~14K cells) were integrated
with scVI using `batch_key='study'`. These populations have strong, discrete
transcriptomic identities that survive standard batch correction.

In [ ]:
tier1_path = INT_DIR / 'tier1_nonresident.h5ad'

if tier1_path.exists():
    adata_t1 = sc.read_h5ad(tier1_path)
    print(f'Tier 1: {adata_t1.shape[0]:,} cells × {adata_t1.shape[1]:,} genes')
    print(f'Studies: {adata_t1.obs["study"].nunique()}')
    print(f'Cell types: {adata_t1.obs["cell_type_final"].nunique()}')
    print()
    print(adata_t1.obs['cell_type_final'].value_counts())
    print()
    print('obsm keys:', list(adata_t1.obsm.keys()))

    # UMAP — unintegrated vs scVI
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    for col_i, (umap_key, label) in enumerate([
        ('X_umap_unintegrated', 'Unintegrated'),
        ('X_umap_scvi', 'scVI integrated'),
    ]):
        if umap_key not in adata_t1.obsm:
            continue
        adata_t1.obsm['X_umap'] = adata_t1.obsm[umap_key]
        for row_i, color_key in enumerate(['study', 'cell_type_final']):
            ax = axes[row_i, col_i]
            sc.pl.umap(adata_t1, color=color_key, ax=ax, show=False,
                       frameon=False, s=8, title=f'{label} — {color_key}')

    # Third column: condition
    if 'X_umap_scvi' in adata_t1.obsm:
        adata_t1.obsm['X_umap'] = adata_t1.obsm['X_umap_scvi']
        if 'condition_harmonized' in adata_t1.obs.columns:
            sc.pl.umap(adata_t1, color='condition_harmonized', ax=axes[0, 2],
                       show=False, frameon=False, s=8, title='scVI — condition')
        if 'leiden_scvi_10' in adata_t1.obs.columns:
            sc.pl.umap(adata_t1, color='leiden_scvi_10', ax=axes[1, 2],
                       show=False, frameon=False, s=8, title='scVI — Leiden 1.0')

    fig.suptitle('Tier 1: Non-Resident Cell Integration', fontsize=14, y=1.01)
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'notebook_05_tier1_umap.png', dpi=150, bbox_inches='tight')
    plt.show()
    del adata_t1
else:
    print('Tier 1 output not found')

## Tier 2 NP — Approach Comparison

NP resident cells (NP subtypes + EP cells) were integrated using four approaches.
Each row below shows UMAPs colored by study, cell type, condition, and compartment.
The key question: which approach preserves the NP cell state continuum
(notochordal → mature → degenerative) while still removing batch effects?

In [ ]:
def plot_approach_grid(adata, compartment, fig_dir):
    """Plot 5-column (unintegrated + 4 approaches) × 4-row (color keys) UMAP grid."""
    umap_keys = []
    umap_labels = []
    for key, label in [
        ('X_umap_unintegrated', 'Unintegrated'),
        ('X_umap_scvi', 'A: scVI'),
        ('X_umap_scanvi', 'B: scANVI'),
        ('X_umap_harmony', 'C: Harmony'),
        ('X_umap_bbknn', 'D: BBKNN'),
    ]:
        if key in adata.obsm:
            umap_keys.append(key)
            umap_labels.append(label)

    color_keys = ['study', 'cell_type_final', 'condition_harmonized', 'compartment']
    color_keys = [c for c in color_keys if c in adata.obs.columns]

    n_cols = len(umap_keys)
    n_rows = len(color_keys)
    if n_cols == 0:
        print(f'  No UMAP embeddings found for {compartment}')
        return

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4.5 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]
    if n_cols == 1:
        axes = axes[:, np.newaxis]

    for col_i, (umap_key, umap_label) in enumerate(zip(umap_keys, umap_labels)):
        adata.obsm['X_umap'] = adata.obsm[umap_key]
        for row_i, color_key in enumerate(color_keys):
            ax = axes[row_i, col_i]
            sc.pl.umap(adata, color=color_key, ax=ax, show=False,
                       frameon=False, s=1, alpha=0.4)
            if row_i == 0:
                ax.set_title(umap_label, fontsize=11, fontweight='bold')
            if col_i == 0:
                ax.set_ylabel(color_key.replace('_', ' '), fontsize=10)

    fig.suptitle(f'Tier 2 {compartment}: Integration Approach Comparison',
                 fontsize=14, y=1.01)
    fig.tight_layout()
    fig.savefig(fig_dir / f'notebook_05_tier2_{compartment}_grid.png',
                dpi=150, bbox_inches='tight')
    plt.show()


np_path = INT_DIR / 'tier2_resident_NP.h5ad'
if np_path.exists():
    adata_np = sc.read_h5ad(np_path)
    print(f'NP: {adata_np.shape[0]:,} cells × {adata_np.shape[1]:,} genes')
    print(f'Studies: {adata_np.obs["study"].nunique()}')
    print(f'Cell types: {adata_np.obs["cell_type_final"].nunique()}')
    print(adata_np.obs['cell_type_final'].value_counts())
    print('obsm keys:', list(adata_np.obsm.keys()))
    print()
    plot_approach_grid(adata_np, 'NP', FIG_DIR)
else:
    adata_np = None
    print('Tier 2 NP output not found')

## Tier 2 AF — Approach Comparison

AF resident cells (AF subtypes + fibroblasts) form the largest group (~281K
cells). The same four approaches are compared. AF cells have a clearer
inner/outer distinction but the mechanical stress subtype may be harder to
preserve under aggressive batch correction.

In [ ]:
af_path = INT_DIR / 'tier2_resident_AF.h5ad'
if af_path.exists():
    adata_af = sc.read_h5ad(af_path)
    print(f'AF: {adata_af.shape[0]:,} cells × {adata_af.shape[1]:,} genes')
    print(f'Studies: {adata_af.obs["study"].nunique()}')
    print(f'Cell types: {adata_af.obs["cell_type_final"].nunique()}')
    print(adata_af.obs['cell_type_final'].value_counts())
    print('obsm keys:', list(adata_af.obsm.keys()))
    print()
    plot_approach_grid(adata_af, 'AF', FIG_DIR)
else:
    adata_af = None
    print('Tier 2 AF output not found')

## Metrics Comparison

Grouped bar charts comparing batch mixing (iLISI, batch ASW) and biological
conservation (cLISI, cell type ASW, isolated label F1) across approaches.
The overall scib score weights batch mixing at 40% and bio conservation at 60%,
reflecting the priority of preserving cell state information in this atlas.

In [ ]:
if len(metrics_df) > 0:
    # Separate Tier 2 metrics for NP and AF
    for compartment in ['NP', 'AF']:
        comp_df = metrics_df[metrics_df['compartment'] == compartment].copy()
        if len(comp_df) < 2:
            print(f'Skipping {compartment}: fewer than 2 approaches')
            continue

        metric_cols = ['iLISI', 'celltype_ASW', 'batch_ASW', 'isolated_label_F1', 'scib_overall']
        available = [c for c in metric_cols if c in comp_df.columns and comp_df[c].notna().any()]
        if not available:
            continue

        fig, ax = plt.subplots(figsize=(10, 5))
        x = np.arange(len(available))
        width = 0.8 / len(comp_df)

        for i, (_, row) in enumerate(comp_df.iterrows()):
            vals = [row.get(c, np.nan) for c in available]
            ax.bar(x + i * width, vals, width, label=row['approach'], alpha=0.8)

        ax.set_xticks(x + width * (len(comp_df) - 1) / 2)
        ax.set_xticklabels(available, rotation=30, ha='right')
        ax.set_ylabel('Score')
        ax.set_title(f'{compartment}: Integration Metrics Comparison')
        ax.legend()
        fig.tight_layout()
        fig.savefig(FIG_DIR / f'notebook_05_metrics_{compartment}.png',
                    dpi=150, bbox_inches='tight')
        plt.show()
else:
    print('No metrics available')

## Continuum Preservation — NP Score Distributions

The four NP signature scores (notochordal, mature chondrocyte, stressed/
degenerative, fibrocartilaginous) capture the cell state continuum. For each
integration approach, we show KDE plots of these scores, split by disease
condition. If integration overcorrects, the score distributions flatten or
lose their condition-associated differences.

In [ ]:
np_score_cols = ['score_NP_notochordal', 'score_NP_mature_chondrocyte',
                 'score_NP_stressed_degenerative', 'score_NP_fibrocartilaginous']

if adata_np is not None:
    available_scores = [c for c in np_score_cols if c in adata_np.obs.columns]
    if available_scores and 'condition_harmonized' in adata_np.obs.columns:
        conditions = sorted(adata_np.obs['condition_harmonized'].dropna().unique())

        fig, axes = plt.subplots(1, len(available_scores),
                                  figsize=(5 * len(available_scores), 5))
        if len(available_scores) == 1:
            axes = [axes]

        for ax, col in zip(axes, available_scores):
            for cond in conditions:
                data = adata_np.obs.loc[
                    adata_np.obs['condition_harmonized'] == cond, col
                ].dropna()
                if len(data) > 10:
                    data.plot.kde(ax=ax, label=cond, alpha=0.7)
            ax.set_title(col.replace('score_', '').replace('_', ' ').title(),
                         fontsize=10)
            ax.set_xlabel('Score')
            ax.legend(fontsize=7)

        fig.suptitle('NP Subtype Scores by Condition (Integrated Data)',
                     fontsize=12, y=1.02)
        fig.tight_layout()
        fig.savefig(FIG_DIR / 'notebook_05_np_continuum.png',
                    dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print('NP score columns or condition not available')
else:
    print('NP data not loaded')

## Cluster Count Comparison

A simple but informative check: how many Leiden clusters (at resolution 0.5)
does each approach produce? If an approach compresses everything into one or
two clusters, that's the "blob" problem. More clusters suggest better
preservation of biological heterogeneity — but too many may indicate failed
batch correction (batch-driven clusters).

In [ ]:
if len(metrics_df) > 0 and 'n_clusters_05' in metrics_df.columns:
    tier2 = metrics_df[metrics_df['tier'] == 'tier2'].copy()
    if len(tier2) > 0 and tier2['n_clusters_05'].notna().any():
        fig, ax = plt.subplots(figsize=(10, 5))

        # Group by compartment
        compartments = tier2['compartment'].unique()
        x_positions = []
        x_labels = []
        colors = []
        palette = {'NP': '#3498db', 'AF': '#e74c3c'}

        pos = 0
        for comp in compartments:
            comp_data = tier2[tier2['compartment'] == comp]
            for _, row in comp_data.iterrows():
                ax.bar(pos, row['n_clusters_05'], color=palette.get(comp, '#95a5a6'),
                       alpha=0.8, edgecolor='white')
                x_positions.append(pos)
                x_labels.append(f"{row['approach']}\n({comp})")
                pos += 1
            pos += 0.5  # gap between compartments

        ax.set_xticks(x_positions)
        ax.set_xticklabels(x_labels, fontsize=9)
        ax.set_ylabel('Number of clusters (resolution 0.5)')
        ax.set_title('Cluster Count by Integration Approach')
        ax.axhline(y=1, color='red', linestyle='--', alpha=0.5, label='Blob threshold')
        ax.legend()

        fig.tight_layout()
        fig.savefig(FIG_DIR / 'notebook_05_cluster_counts.png',
                    dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print('No cluster count data in Tier 2 metrics')
else:
    print('Cluster count data not available')

## Condition Separability

A logistic regression classifier (5-fold CV) was trained on each integrated
embedding to distinguish healthy from degenerated cells. An accuracy well above
chance (~50%) indicates that condition-associated transcriptomic variation
survived integration. Very high accuracy (>90%) might indicate that the
condition signal dominates — which is biologically desirable but should be
verified.

In [ ]:
if len(metrics_df) > 0 and 'condition_accuracy' in metrics_df.columns:
    tier2 = metrics_df[metrics_df['tier'] == 'tier2'].copy()
    if len(tier2) > 0 and tier2['condition_accuracy'].notna().any():
        fig, ax = plt.subplots(figsize=(10, 5))

        compartments = tier2['compartment'].unique()
        palette = {'NP': '#3498db', 'AF': '#e74c3c'}
        pos = 0
        x_positions = []
        x_labels = []

        for comp in compartments:
            comp_data = tier2[tier2['compartment'] == comp]
            for _, row in comp_data.iterrows():
                ax.bar(pos, row['condition_accuracy'],
                       color=palette.get(comp, '#95a5a6'),
                       alpha=0.8, edgecolor='white')
                x_positions.append(pos)
                x_labels.append(f"{row['approach']}\n({comp})")
                pos += 1
            pos += 0.5

        ax.set_xticks(x_positions)
        ax.set_xticklabels(x_labels, fontsize=9)
        ax.set_ylabel('Accuracy')
        ax.set_title('Condition Separability (5-Fold CV Logistic Regression)')
        ax.axhline(y=0.5, color='grey', linestyle='--', alpha=0.5, label='Chance')
        ax.axhline(y=0.6, color='green', linestyle='--', alpha=0.5, label='60% threshold')
        ax.set_ylim(0, 1)
        ax.legend()

        fig.tight_layout()
        fig.savefig(FIG_DIR / 'notebook_05_condition_accuracy.png',
                    dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print('No condition accuracy data in Tier 2 metrics')
else:
    print('Condition accuracy not available')

## Full Metrics Table

Complete metrics for all tiers, compartments, and approaches. The "best"
approach per compartment is highlighted by the highest `scib_overall` score
(40% batch mixing + 60% biological conservation).

In [ ]:
if len(metrics_df) > 0:
    # Highlight best per compartment
    display_cols = ['tier', 'compartment', 'approach', 'iLISI', 'cLISI',
                    'batch_ASW', 'celltype_ASW', 'isolated_label_F1',
                    'scib_overall', 'n_clusters_05', 'condition_accuracy']
    available_cols = [c for c in display_cols if c in metrics_df.columns]
    display_df = metrics_df[available_cols].copy()

    # Round numeric columns
    for col in display_df.select_dtypes(include='number').columns:
        display_df[col] = display_df[col].round(3)

    def highlight_best(s):
        if s.name != 'scib_overall' or s.isna().all():
            return ['' for _ in s]
        is_max = s == s.max()
        return ['background-color: #d5f5e3' if v else '' for v in is_max]

    display(
        display_df.style
        .apply(highlight_best, axis=0)
        .set_caption('Integration Metrics Summary')
    )
else:
    print('No metrics to display')

## Notes and Human Checkpoint

### Approach stubs (not run)
- **Approach E** (metacell aggregation): documented fallback if single-cell
  integration consistently overcorrects.
- **Approach F** (label transfer without forced integration): most conservative
  fallback — each dataset keeps its own UMAP.

### Questions for review
1. Which integration approach (A–D) best preserves cell state variation while
   adequately removing batch effects?
2. Is any approach clearly superior, or is a combination needed (e.g., scANVI
   for NP, Harmony for AF)?
3. Does the "blob" problem recur with any approach? If so, is Approach E or F
   the appropriate fallback?
4. Should the analysis proceed with integrated data, per-dataset data, or both
   in parallel?
5. Are there any study-specific effects that persist after integration and need
   to be addressed as covariates in downstream DE analysis?
6. Does the integration reveal any new cell states not visible in per-dataset
   analysis?

In [ ]:
# Clean up large objects
for var in ['adata_np', 'adata_af']:
    if var in dir():
        exec(f'del {var}')

saved_figures = sorted(FIG_DIR.glob('notebook_05_*.png'))
print('Saved figures:')
for f in saved_figures:
    print(f'  {f.relative_to(BASE)}')